# 21.6 Polars:单机上的"大数据"利器 / Polars: Big-Data Power on a Single Machine

**中文**:前面几节反复出现一个诚实的主线:*"数据能进单机内存时,别用 Spark/Dask,单机引擎更快。"* 那**单机上最快的引擎**是什么?**Polars** 是当下的答案。它用 **Rust** 写成,天生:①**列式存储(基于 Apache Arrow)**——同列数据连续存放,缓存友好、可 SIMD 向量化;②**多线程并行**——自动吃满你所有 CPU 核心(pandas 基本单线程);③**惰性执行 + 查询优化器**——像 Spark 的 Catalyst 一样做谓词/列裁剪下推。结果是:处理几千万甚至上亿行,**Polars 常比 pandas 快 5–30 倍**,而且很多"以为需要 Spark 集群"的活,一台笔记本用 Polars 就搞定了。本节用**真实基准测试**(不是模拟!)量化 Polars vs pandas 的差距,并展示它的惰性优化器。
**English**: An honest thread recurred in prior sections: *"when data fits in single-machine memory, don't use Spark/Dask — single-machine engines are faster."* So **what is the fastest single-machine engine?** **Polars** is today's answer. Written in **Rust**, it is inherently: ① **columnar (built on Apache Arrow)** — same-column data stored contiguously, cache-friendly and SIMD-vectorizable; ② **multi-threaded** — automatically saturates all your CPU cores (pandas is basically single-threaded); ③ **lazy execution + query optimizer** — pushes down predicates/column pruning like Spark's Catalyst. The result: processing tens of millions or even hundreds of millions of rows, **Polars is often 5–30x faster than pandas**, and many jobs "you thought needed a Spark cluster" are done on a single laptop with Polars. This section uses **real benchmarks** (not simulations!) to quantify Polars vs pandas, and shows its lazy optimizer.

---

**中文**:**为什么 Polars 这么快?**(面试爱问的"底层原理"):
**English**: **Why is Polars so fast?** (the "underlying principles" interviewers love):
1. **中文**:**列式 + Arrow 内存**:分析查询通常只碰几列(如 `groupby cat, mean(val)`)。列式存储让这几列**连续读入 CPU 缓存**,而行式(pandas 的部分场景/Python 对象)会读入一堆用不到的数据、缓存命中差。
   **Columnar + Arrow memory**: analytical queries usually touch only a few columns (e.g. `groupby cat, mean(val)`). Columnar storage reads those columns **contiguously into CPU cache**, whereas row-based layouts read lots of unused data with poor cache hits.
2. **中文**:**多线程 + SIMD**:Polars 自动把工作切给所有 CPU 核心并行,还用 SIMD 指令一次处理多个值。pandas 绝大多数操作是**单线程**的——你的 8 核机器只用了 1 核。
   **Multi-threaded + SIMD**: Polars automatically splits work across all CPU cores and uses SIMD to process multiple values at once. Most pandas operations are **single-threaded** — your 8-core machine uses just 1 core.
3. **中文**:**惰性 + 查询优化**:`pl.scan_*` / `.lazy()` 构建查询计划,优化器做**谓词下推、列裁剪(只读需要的列)、投影下推**,甚至能**流式处理超过内存的数据**(streaming engine)。
   **Lazy + query optimization**: `pl.scan_*` / `.lazy()` builds a query plan; the optimizer does **predicate pushdown, column pruning (read only needed columns), projection pushdown**, and can even **stream data larger than memory** (streaming engine).

> 💡 **面试速查 / Interview cheat-sheet（★★ 现代数据栈热点）**
> **中文**:**Polars**=Rust 写的高性能 DataFrame。**快的三大原因**:①列式(Arrow, 缓存友好+SIMD);②多线程(吃满所有核, pandas 基本单线程);③惰性+查询优化(谓词/列裁剪下推, streaming 处理超内存)。**两种 API**:eager(即时, 像 pandas)、**lazy**(`.lazy()...collect()`, 走优化器, 推荐)。**表达式 API**:`pl.col("x")` 表达式可组合/并行, 是 Polars 的精髓(比 pandas 的 apply 快得多)。**vs pandas**:5–30x 快、内存省、语法更一致(无 index 之乱)、原生缺失值; 但生态没 pandas 老(可视化/ML 库多要 pandas)。**vs Spark/Dask**:Polars 是**单机**(但能处理到内存几倍大, streaming); 真超单机(TB+/多机)才需 Spark。**何时用**:单机几百 MB~几十 GB 的分析/ETL——现在的首选。互操作:`.to_pandas()`/`.to_arrow()` 零拷贝换 pandas/其他 Arrow 工具。面试金句:*"Polars 靠列式Arrow+多线程+惰性查询优化, 在单机上比 pandas 快 5–30 倍; 用 lazy API 和表达式让优化器做谓词/列裁剪下推、甚至流式超内存处理; 数据能进单机(或几倍内存)就用 Polars/DuckDB, 别动辄上 Spark。"*
> **English**: **Polars** = a high-performance DataFrame written in Rust. **Three reasons it's fast**: ① columnar (Arrow, cache-friendly + SIMD); ② multi-threaded (saturates all cores; pandas is basically single-threaded); ③ lazy + query optimization (predicate/column-pruning pushdown, streaming for larger-than-memory). **Two APIs**: eager (immediate, pandas-like), **lazy** (`.lazy()...collect()`, goes through the optimizer, recommended). **Expression API**: `pl.col("x")` expressions are composable/parallel, the essence of Polars (far faster than pandas `apply`). **vs pandas**: 5–30x faster, less memory, more consistent syntax (no index mess), native nulls; but a younger ecosystem (viz/ML libs often need pandas). **vs Spark/Dask**: Polars is **single-machine** (but handles several-times-memory via streaming); true over-single-machine (TB+/multi-machine) needs Spark. **When to use**: single-machine hundreds-of-MB to tens-of-GB analytics/ETL — today's default. Interop: `.to_pandas()`/`.to_arrow()` zero-copy to pandas/other Arrow tools. Interview line: *"Polars uses columnar Arrow + multithreading + lazy query optimization to run 5–30x faster than pandas on a single machine; use the lazy API and expressions so the optimizer does predicate/column-pruning pushdown and even streams larger-than-memory data; if data fits single-machine (or a few times memory) use Polars/DuckDB, don't reach for Spark."*


In [ ]:

# ============================================================
# 真实基准测试:pandas vs Polars(eager)vs Polars(lazy)/ REAL benchmark
# 中文:8 百万行, 典型分析负载:过滤 val>50 → 按 cat 分组 → 求 mean 和 count。计时对比(取多次最小值)。
# English: 8M rows, typical analytical workload: filter val>50 → group by cat → mean & count. Timed (min of repeats).
# ============================================================
import numpy as np, pandas as pd, polars as pl, time
np.random.seed(0)
N=8_000_000
keys=np.random.randint(0,1000,N); cat=np.random.choice(["a","b","c","d"],N); val=np.random.rand(N)*100
pdf=pd.DataFrame({"key":keys,"cat":cat,"val":val})       # pandas DataFrame
pldf=pl.DataFrame({"key":keys,"cat":cat,"val":val})      # Polars DataFrame(同样的数据)/ same data
print(f"数据规模 / rows: {N:,}   Polars 使用线程数 / threads: {pl.thread_pool_size()}")

def bench(f,rep=3): return min(( (lambda t0=time.time(): (f(),time.time()-t0)[1])() ) for _ in range(rep))
# pandas(基本单线程)/ pandas (mostly single-threaded)
t_pd=bench(lambda: pdf[pdf.val>50].groupby("cat")["val"].agg(["mean","count"]))
# Polars eager(多线程, 列式)/ Polars eager (multi-threaded, columnar)
t_pl=bench(lambda: pldf.filter(pl.col("val")>50).group_by("cat").agg(
                    pl.col("val").mean().alias("mean"), pl.len().alias("count")))
# Polars lazy(+ 查询优化器)/ Polars lazy (+ query optimizer)
t_lz=bench(lambda: pldf.lazy().filter(pl.col("val")>50).group_by("cat").agg(
                    pl.col("val").mean().alias("mean"), pl.len().alias("count")).collect())
print(f"\npandas        : {t_pd*1000:7.1f} ms   (基准 baseline)")
print(f"Polars eager  : {t_pl*1000:7.1f} ms   ({t_pd/t_pl:4.1f}x 更快 faster)")
print(f"Polars lazy   : {t_lz*1000:7.1f} ms   ({t_pd/t_lz:4.1f}x 更快 faster)")


In [ ]:

# ============================================================
# 惰性查询优化器:看它自动做列裁剪/谓词下推 / lazy optimizer: column pruning & predicate pushdown
# ============================================================
plan = (pldf.lazy()
            .filter(pl.col("val")>50)                    # 过滤 / filter
            .group_by("cat").agg(pl.col("val").mean()))  # 分组聚合 / group-agg
print("优化后的查询计划 / optimized query plan:")
print(plan.explain())
print("\n观察:计划底部 PROJECT[...] 只读 2/3 列(cat,val)——用不到的 key 列根本不读(列裁剪);")
print("     FILTER 被下推到扫描阶段——和 21.2 的 Catalyst/DuckDB 是同一套优化思想。")

# 表达式 API:Polars 的精髓(可组合、并行、比 pandas apply 快得多)/ the Expression API
demo = (pldf.lazy()
            .with_columns([(pl.col("val")*2).alias("val2"),                     # 新列 / new column
                           pl.col("val").rank().over("cat").alias("rank_in_cat")]) # 分组内排名 / rank within group
            .filter(pl.col("val2")>150)
            .select(["cat","val","val2","rank_in_cat"])
            .head(3).collect())
print("\n表达式 API 示例(分组内排名 + 派生列, 全部并行执行)/ expression API demo:")
print(demo)


In [ ]:

# ============================================================
# 可视化:基准对比 / benchmark visualization
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
names=["pandas","Polars\neager","Polars\nlazy"]; times=[t_pd*1000,t_pl*1000,t_lz*1000]; cols=["#C44E52","#55A868","#4C72B0"]
b=ax[0].bar(names,times,color=cols)
for bar,tm in zip(b,times): ax[0].text(bar.get_x()+bar.get_width()/2,tm+1,f"{tm:.0f}ms",ha="center",fontsize=11,weight="bold")
ax[0].set_ylabel("耗时 ms(越低越好)"); ax[0].set_title(f"8百万行 groupby: Polars 比 pandas 快 {t_pd/t_lz:.1f}x")
# speedup factors
ax[1].bar(names,[1,t_pd/t_pl,t_pd/t_lz],color=cols)
for i,s in enumerate([1,t_pd/t_pl,t_pd/t_lz]): ax[1].text(i,s+0.1,f"{s:.1f}x",ha="center",fontsize=11,weight="bold")
ax[1].set_ylabel("相对 pandas 的加速倍数"); ax[1].set_title("加速倍数(列式+多线程+优化器)")
plt.tight_layout(); plt.savefig("/tmp/big06_viz.png",dpi=80); plt.show()
print(f"Polars 用列式内存+多线程({pl.thread_pool_size()}线程)+惰性优化器, 在单机上就实现了数倍加速")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **"大数据必须上集群"是过时的直觉**:这次是**真实基准**——同样一台机器、同样 800 万行数据,Polars 比 pandas 快 5 倍左右,而且这个差距在数据更大、列更多、操作更复杂时会拉到 10–30 倍。原因是纯粹的工程优势:列式内存(缓存友好)+ 多线程(吃满所有核,而 pandas 只用 1 核)+ 查询优化器(只读需要的列)。**很多公司花大价钱搭 Spark 集群处理的几千万行数据,其实一台配置不错的笔记本用 Polars 就能秒级处理**——这不是夸张,是行业正在发生的真实转变(现代数据栈 modern data stack)。
2. **惰性 + 表达式是 Polars 的正确用法**:①**惰性(lazy)**:用 `.lazy()...collect()` 或 `pl.scan_parquet`,让优化器先看到完整查询再执行——它会自动做列裁剪(计划里 `PROJECT 2/3 COLUMNS` 就是只读 2 列)、谓词下推,和 21.2 的 Catalyst/DuckDB 是同一套思想。②**表达式 API**(`pl.col(...)`):这是 Polars 的灵魂,表达式可组合、可并行,一句 `rank().over("cat")` 就并行算出分组内排名——而 pandas 里 `groupby.apply` 往往慢几十倍。写 Polars 要**用表达式思维,而不是把 pandas 的 for 循环/apply 直译过来**。
3. **诚实的边界:Polars 不是万能替代,pandas 生态仍然重要**。①**生态成熟度**:pandas 有 20 年积累,几乎所有可视化、ML、教程都默认 pandas;Polars 虽在追赶,但很多库还只吃 pandas DataFrame——好在 `.to_pandas()` / `.to_arrow()` 常常是**零拷贝**的,可以"用 Polars 做重活、转 pandas 交给下游库"。②**不是所有场景都快那么多**:极小数据(几千行)上,Polars 的启动/优化开销让它未必比 pandas 快;需要复杂的、Polars 还没实现的操作时,pandas 更灵活。③**它仍是单机的**:Polars 能靠 streaming 处理**超过内存几倍**的数据,但真到 TB 级、需要几百台机器时,还是得 Spark。**结论:把 Polars(和下节的 DuckDB)作为单机数据处理的新默认,pandas 用于生态兼容和小数据,Spark 留给真正的超大规模——这就是 2020 年代数据工程师该有的工具地图。**

**English**:
1. **"Big data must go to a cluster" is an outdated intuition**: this time it's a **real benchmark** — same machine, same 8M rows, Polars ~5x faster than pandas, and this gap widens to 10–30x with bigger data, more columns, more complex operations. The reason is pure engineering advantage: columnar memory (cache-friendly) + multithreading (saturate all cores, while pandas uses 1) + a query optimizer (read only needed columns). **The tens-of-millions-of-rows data many companies spend heavily on Spark clusters to process can actually be handled in seconds on a decent laptop with Polars** — no exaggeration, but a real industry shift underway (the modern data stack).
2. **Lazy + expressions is the correct way to use Polars**: ① **lazy**: use `.lazy()...collect()` or `pl.scan_parquet` so the optimizer sees the full query before executing — it auto-does column pruning (the plan's `PROJECT 2/3 COLUMNS` means reading only 2 columns) and predicate pushdown, the same idea as 21.2's Catalyst/DuckDB. ② **the expression API** (`pl.col(...)`): the soul of Polars, expressions are composable and parallel — one `rank().over("cat")` computes within-group ranks in parallel, whereas pandas `groupby.apply` is often tens of times slower. Write Polars with **expression thinking, not by literally translating pandas for-loops/apply**.
3. **Honest limits: Polars is not an omnipotent replacement, and pandas's ecosystem still matters**. ① **Ecosystem maturity**: pandas has 20 years behind it, and nearly all viz, ML, and tutorials default to pandas; Polars is catching up, but many libraries only accept pandas DataFrames — fortunately `.to_pandas()` / `.to_arrow()` is often **zero-copy**, so "do the heavy lifting in Polars, convert to pandas for downstream libraries." ② **Not always dramatically faster**: on tiny data (a few thousand rows), Polars's startup/optimization overhead may make it no faster than pandas; for complex operations Polars hasn't implemented, pandas is more flexible. ③ **It's still single-machine**: Polars can stream data **several times larger than memory**, but at true TB scale needing hundreds of machines, you still need Spark. **Conclusion: make Polars (and the next section's DuckDB) the new default for single-machine data processing, use pandas for ecosystem compatibility and small data, and reserve Spark for truly huge scale — this is the tool map a 2020s data engineer should have.**

> 💼 **实战视角 / Practical angle**
> **中文**:Polars 落地:①**日常 ETL/分析**用 `pl.scan_csv/scan_parquet(...).filter(...).group_by(...).collect()`(惰性+流式, 只读需要的列/行);②**写表达式**而非 apply——`pl.col()`、`.over()`(窗口)、`.list`/`.str` 命名空间;③**大文件用 lazy + streaming**(`collect(streaming=True)`)处理超内存数据;④**和 pandas/Arrow 互操作** `.to_pandas()/.to_arrow()`(常零拷贝), 重活给 Polars、交下游库时转 pandas;⑤读写 Parquet(下节)是最佳拍档。**选型**:单机分析首选 Polars/DuckDB, 生态兼容用 pandas, 超大规模才 Spark。面试金句:*"Polars 用列式Arrow+多线程+惰性优化器在单机上比 pandas 快数倍到数十倍; 正确用法是 lazy API + 表达式让优化器做列裁剪谓词下推、streaming 处理超内存; 它是单机新默认, 和 DuckDB 搭 Parquet 覆盖绝大多数'中等大数据', 真 TB+ 才上 Spark。"*
> **English**: Polars in practice: ① **daily ETL/analytics** with `pl.scan_csv/scan_parquet(...).filter(...).group_by(...).collect()` (lazy + streaming, read only needed columns/rows); ② **write expressions** not apply — `pl.col()`, `.over()` (windows), `.list`/`.str` namespaces; ③ **big files use lazy + streaming** (`collect(streaming=True)`) for larger-than-memory data; ④ **interop with pandas/Arrow** `.to_pandas()/.to_arrow()` (often zero-copy) — heavy work in Polars, convert to pandas for downstream libs; ⑤ read/write Parquet (next section) is the perfect pairing. **Tool choice**: Polars/DuckDB first for single-machine analytics, pandas for ecosystem compatibility, Spark only for huge scale. Interview line: *"Polars uses columnar Arrow + multithreading + a lazy optimizer to run several to tens of times faster than pandas on a single machine; the right usage is the lazy API + expressions so the optimizer does column-pruning/predicate pushdown and streams larger-than-memory data; it's the new single-machine default, and paired with DuckDB + Parquet covers most 'medium big data' — reach for Spark only at true TB+."*

---
### 小结 / Summary
- **中文**:Polars(Rust)靠列式Arrow + 多线程 + 惰性查询优化, 单机上比 pandas 快 5–30x(本节真实基准 ~5x)。
- **English**: Polars (Rust) uses columnar Arrow + multithreading + lazy query optimization to run 5–30x faster than pandas on a single machine (real benchmark here ~5x).
- **中文**:用 lazy API + 表达式(pl.col)让优化器做列裁剪/谓词下推, 甚至 streaming 处理超内存数据。
- **English**: Use the lazy API + expressions (pl.col) so the optimizer does column-pruning/predicate pushdown, even streaming larger-than-memory data.
- **中文**:单机分析新默认(配 DuckDB+Parquet); pandas 保生态兼容, 真 TB+/多机才用 Spark。
- **English**: The new single-machine default (with DuckDB + Parquet); pandas for ecosystem compatibility, Spark only for true TB+/multi-machine.
